# R5 gerçek dosyalarını özel Google Drive klasörlerine aktar
Bu defter ChatGPT içinde çalıştırılmadı; mevcut bir Drive yükleme makbuzu değildir.
Tek hücreyi Google Colab’da çalıştırın, gerçek R5 ZIP dosyasını seçin ve Primary Google hesabınızla OAuth onayı verin. Kod önce paket içeriğini/hashlerini denetler; eski boş R4 kapanış dosyasını reddeder. Tam arşivi ve dil/model/veri alt paketlerini ayrı kanonik klasörlere yükler; geri indirip SHA-256 karşılaştırır. Önceki farklı dosyaları ezmez, paylaşım izni açmaz, parola veya token yazdırmaz.
Modelin geniş kapsamlı yeterliliği paket/aktarım bütünlüğünden ayrıdır; kaynak kartındaki sınırlar korunur.


In [ ]:
from pathlib import Path
import tempfile, sys
from google.colab import files
print('ICL_R5_DOGRULANMIS_TESLIM_20261013.zip dosyasını seçin. Başka içerik yüklenmez.')
uploaded = files.upload()
if len(uploaded) != 1:
    raise RuntimeError('Tek bir R5 ZIP dosyası seçilmeli.')
name, content = next(iter(uploaded.items()))
work = Path(tempfile.mkdtemp(prefix='icl-r5-authorized-transfer-'))
archive = work / 'ICL_R5_DOGRULANMIS_TESLIM_20261013.zip'
archive.write_bytes(content)
(work / "verify_archive.py").write_text('"""Validate the actual R5 payload, without importing any code from the archive.\nSelf-supplied hashes detect corruption, not authorship or scientific correctness.\n"""\nfrom pathlib import Path,PurePosixPath\nimport hashlib,json,stat,zipfile,math\nPREFIX=\'ICL_R5_20261013/\'\ndef strict_json(raw):\n def pairs(xs):\n  d={}\n  for k,v in xs:\n   if k in d:raise ValueError(\'DUPLICATE_JSON_KEY\')\n   d[k]=v\n  return d\n def finite_float(s):\n  v=float(s)\n  if not math.isfinite(v):raise ValueError(\'NONFINITE\')\n  return v\n return json.loads(raw,object_pairs_hook=pairs,parse_float=finite_float,parse_constant=lambda s:(_ for _ in ()).throw(ValueError(\'NONFINITE\')))\ndef validate(path):\n with zipfile.ZipFile(path) as z:\n  infos=z.infolist();names=[i.filename for i in infos]\n  if len(names)!=len(set(names)):raise ValueError(\'DUPLICATE_ARCHIVE_MEMBER\')\n  if len(names)>20000 or sum(i.file_size for i in infos)>1_000_000_000:raise ValueError(\'ARCHIVE_SIZE_LIMIT\')\n  for i in infos:\n   p=PurePosixPath(i.filename)\n   if p.as_posix()!=i.filename or i.is_dir():raise ValueError(\'NONCANONICAL_ARCHIVE_MEMBER\')\n   if not i.filename.startswith(PREFIX) or p.is_absolute() or \'..\' in p.parts or \'\\\\\' in i.filename or \':\' in i.filename or stat.S_ISLNK(i.external_attr>>16) or i.flag_bits&1:raise ValueError(\'UNSAFE_ARCHIVE_MEMBER\')\n   if i.file_size>250_000_000:raise ValueError(\'MEMBER_SIZE_LIMIT\')\n  man=strict_json(z.read(PREFIX+\'MANIFEST.json\'))\n  if man.get(\'schema\')!=\'ICL_R5_RELEASE_MANIFEST/1\' or not isinstance(man.get(\'files\'),dict):raise ValueError(\'MANIFEST_SCHEMA\')\n  expected={PREFIX+p for p in man[\'files\']}|{PREFIX+\'MANIFEST.json\'}\n  if set(names)!=expected:raise ValueError(\'UNEXPECTED_OR_MISSING_MEMBER\')\n  for p,meta in man[\'files\'].items():\n   n=PREFIX+p\n   if z.getinfo(n).file_size!=meta[\'bytes\']:raise ValueError(\'SIZE_MISMATCH:\'+p)\n   h=hashlib.sha256()\n   with z.open(n) as f:\n    for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n   if h.hexdigest()!=meta[\'sha256\']:raise ValueError(\'HASH_MISMATCH:\'+p)\n  required=[\'source/icl_core/icl/frontend.py\',\'source/icl_native/graph.py\',\'models/offline_native_17/weights.json\',\'models/native_17/best/weights.safetensors\',\'models/native_29/best/weights.safetensors\',\'models/native_43/best/weights.safetensors\',\'reports/ALL_TRAINING_RUNS.json\',\'reports/EVAL_SUMMARY.json\',\'run_local.py\',\'run_console.py\']\n  if any(p not in man[\'files\'] or man[\'files\'][p][\'bytes\']==0 for p in required):raise ValueError(\'REAL_MODEL_PAYLOAD_MISSING\')\n  weights=strict_json(z.read(PREFIX+\'models/offline_native_17/weights.json\'))\n  if set(weights)!={\'w1\',\'b1\',\'w2\',\'b2\',\'wo\',\'bo\'} or sum(len(v.get(\'data\',[])) for v in weights.values())!=2774:raise ValueError(\'WEIGHT_PAYLOAD_INVALID\')\n  reports=strict_json(z.read(PREFIX+\'reports/ALL_TRAINING_RUNS.json\'))\n  if len(reports)!=18 or any(not r.get(\'weights_changed\') for r in reports):raise ValueError(\'TRAINING_RECORD_MISSING\')\n  return {\'payload_files\':len(man[\'files\']),\'payload_bytes\':sum(m[\'bytes\'] for m in man[\'files\'].values()),\'model_parameters_offline\':2774,\'training_records\':len(reports),\'has_real_r5_payload\':True,\'scope\':\'integrity and presence, not publisher authentication or full scientific validation\'}\nif __name__==\'__main__\':\n import sys\n print(json.dumps(validate(Path(sys.argv[1])),indent=2))\n', encoding="utf-8")
(work / "upload_drive.py").write_text('"""Explicit user OAuth upload for Google Colab. No passwords/tokens are printed.\nThis is a user-run transfer utility, NOT evidence of an upload by ChatGPT.\nOnly new private version folders are created. Existing different bytes are never overwritten.\n"""\nfrom pathlib import Path\nimport hashlib,json,os,tempfile,zipfile,time,shutil\nfrom verify_archive import validate,PREFIX\nPARENTS={\'control\':\'1cE4POvzQkrhGZeEXEtdyfOaQXKybvL70\',\'core\':\'1l051Boq6aF9L8QzY0lCy0mJrVd4r1hBv\',\'model\':\'1UjNgl7NEFYAMUMEYOX9a_0_9fGWndNMk\',\'registry\':\'1sIbxe8NCS4nDfOeHbhnCQjDa_JDFPCT5\'}\nVERSION=\'2026-10-13_R5_VERIFIED_LOCAL_DELIVERY\'\nEXPECTED_ACCOUNT=\'houyinkyoma357@gmail.com\'\ndef sha(p):\n h=hashlib.sha256()\n with open(p,\'rb\') as f:\n  for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef qquote(s):return s.replace(\'\\\\\',\'\\\\\\\\\').replace("\'","\\\\\'")\ndef main(archive):\n archive=Path(archive);local=validate(archive);print(\'Yerel gerçek içerik doğrulandı:\',local)\n from google.colab import auth\n auth.authenticate_user()\n import google.auth\n from googleapiclient.discovery import build\n from googleapiclient.http import MediaFileUpload,MediaIoBaseDownload\n credentials,_=google.auth.default(scopes=[\'https://www.googleapis.com/auth/drive\']);service=build(\'drive\',\'v3\',credentials=credentials,cache_discovery=False)\n user=service.about().get(fields=\'user(emailAddress)\').execute()[\'user\'][\'emailAddress\']\n if user.lower()!=EXPECTED_ACCOUNT.lower():raise RuntimeError(\'Yanlış hesap. Primary Google hesabıyla yetkilendir; hiçbir dosya yüklenmedi.\')\n def metadata(fid):return service.files().get(fileId=fid,fields=\'id,name,mimeType,size,parents,shared\',supportsAllDrives=True).execute()\n for parent in PARENTS.values():\n  m=metadata(parent)\n  if m.get(\'mimeType\')!=\'application/vnd.google-apps.folder\' or m.get(\'shared\'):raise RuntimeError(\'Özel hedef klasör doğrulanamadı; paylaşım ayarı değiştirilmeden duruldu.\')\n def find(parent,name):\n  q=f"\'{qquote(parent)}\' in parents and trashed=false and name=\'{qquote(name)}\'";found=[];token=None\n  while True:\n   r=service.files().list(q=q,fields=\'files(id,name,mimeType,size),nextPageToken\',pageSize=100,pageToken=token,supportsAllDrives=True,includeItemsFromAllDrives=True).execute();found.extend(r.get(\'files\',[]));token=r.get(\'nextPageToken\')\n   if not token:break\n  if len(found)>1:raise RuntimeError(\'Aynı adlı birden fazla hedef var; tahmin edilmedi: \'+name)\n  return found[0] if found else None\n def folder(parent,name):\n  old=find(parent,name)\n  if old:\n   if old[\'mimeType\']!=\'application/vnd.google-apps.folder\':raise RuntimeError(\'Klasör/dosya adı çakışması\')\n   return old[\'id\']\n  return service.files().create(body={\'name\':name,\'mimeType\':\'application/vnd.google-apps.folder\',\'parents\':[parent]},fields=\'id\',supportsAllDrives=True).execute()[\'id\']\n targets={k:folder(v,VERSION) for k,v in PARENTS.items()};receipts=[]\n def remote_hash(fid):\n  with tempfile.TemporaryFile() as f:\n   dl=MediaIoBaseDownload(f,service.files().get_media(fileId=fid,supportsAllDrives=True),chunksize=8*1024*1024);done=False\n   while not done:_,done=dl.next_chunk(num_retries=3)\n   f.seek(0);h=hashlib.sha256()\n   for b in iter(lambda:f.read(1024*1024),b\'\'):h.update(b)\n   return h.hexdigest()\n def put(path,parent):\n  path=Path(path);digest=sha(path);old=find(parent,path.name)\n  if old:\n   if int(old.get(\'size\',\'-1\'))!=path.stat().st_size or remote_hash(old[\'id\'])!=digest:raise RuntimeError(\'Mevcut farklı dosyanın üzerine yazılmadı: \'+path.name)\n   fid=old[\'id\'];mode=\'ALREADY_PRESENT_READBACK_VERIFIED\'\n  else:\n   req=service.files().create(body={\'name\':path.name,\'parents\':[parent],\'appProperties\':{\'icl_release\':\'R5\',\'local_sha256\':digest}},media_body=MediaFileUpload(str(path),resumable=True,chunksize=8*1024*1024),fields=\'id,size\',supportsAllDrives=True);response=None\n   while response is None:_,response=req.next_chunk(num_retries=3)\n   fid=response[\'id\'];mode=\'UPLOADED_READBACK_VERIFIED\'\n   if remote_hash(fid)!=digest:raise RuntimeError(\'Uzak SHA-256 eşleşmedi: \'+path.name)\n  m=metadata(fid)\n  if m.get(\'shared\') or parent not in m.get(\'parents\',[]):raise RuntimeError(\'Uzak paylaşım/üst klasör denetimi başarısız\')\n  receipt={\'name\':path.name,\'id\':fid,\'parent_id\':parent,\'bytes\':path.stat().st_size,\'sha256\':digest,\'readback_match\':True,\'mode\':\'CONTENT_READBACK_VERIFIED\',\'url\':\'https://drive.google.com/file/d/\'+fid+\'/view\'};receipts.append(receipt)\n  print(path.name,\'→ doğrulandı\');return receipt\n with tempfile.TemporaryDirectory(prefix=\'icl-r5-transfer-\') as td:\n  td=Path(td)\n  # Validation above covers path, membership, symlink, size and hash before extraction.\n  with zipfile.ZipFile(archive) as z:z.extractall(td)\n  r=td/\'ICL_R5_20261013\';normalized=td/\'ICL_R5_DOGRULANMIS_TESLIM_20261013.zip\';shutil.copy2(archive,normalized);put(normalized,targets[\'control\'])\n  def subzip(name,predicate):\n   p=td/name\n   with zipfile.ZipFile(p,\'w\',zipfile.ZIP_DEFLATED) as z:\n    for f in sorted(r.rglob(\'*\')):\n     if f.is_file() and predicate(f.relative_to(r).as_posix()):\n      info=zipfile.ZipInfo(f.relative_to(r).as_posix(),date_time=(2026,10,13,0,0,0));info.external_attr=0o100644<<16;info.compress_type=zipfile.ZIP_DEFLATED;z.writestr(info,f.read_bytes())\n   return p\n  core=subzip(\'ICL_CORE_0.2.1_R5_SOURCE.zip\',lambda n:n.startswith(\'source/icl_core/\') or n in (\'LANGUAGE_HANDOFF.json\',\'README_TR.md\'))\n  model=subzip(\'ICL_NATIVE_R5_MODELS_CODE_EVIDENCE.zip\',lambda n:n.startswith((\'source/icl_native/\',\'models/\',\'reports/\',\'examples/\')) or n in (\'run_local.py\',\'run_console.py\',\'MODEL_CARD.md\',\'README_TR.md\'))\n  data=subzip(\'ICL_R5_DATA_AND_REFERENCE_AUDIT.zip\',lambda n:n.startswith(\'data/\'))\n  put(core,targets[\'core\']);put(r/\'source/icl_core/dist/invariant_cognition_language-0.2.1-py3-none-any.whl\',targets[\'core\']);put(r/\'LANGUAGE_HANDOFF.json\',targets[\'core\'])\n  put(model,targets[\'model\']);put(data,targets[\'model\']);put(r/\'MODEL_CARD.md\',targets[\'model\']);put(r/\'reports/EVAL_SUMMARY.json\',targets[\'model\'])\n  for name in (\'SONUC_RAPORU_TR.md\',\'CURRENT_STATUS.json\',\'TASK_LEDGER.json\',\'MANIFEST.json\',\'PROJECT_MEMORY.md\'):put(r/name,targets[\'control\'])\n  body={\'schema\':\'ICL_R5_DRIVE_READBACK/1\',\'account_verified\':True,\'folders\':targets,\'local_archive_sha256\':sha(archive),\'assets\':receipts.copy(),\'all_assets_readback_verified\':True,\'model_scope\':\'bounded semantic opcode kernel; original broad model gates not automatically complete\',\'public_release\':False,\'permissions_changed\':False}\n  p=Path.cwd()/\'ICL_R5_DRIVE_READBACK.json\';p.write_text(json.dumps(body,indent=2)+\'\\n\');final=put(p,targets[\'registry\']);print(\'Aktarım bitti. Makbuz:\',final[\'url\'])\n return body\nif __name__==\'__main__\':\n import sys\n main(sys.argv[1])\n', encoding="utf-8")
sys.path.insert(0, str(work))
from upload_drive import main
receipt = main(archive)
files.download("ICL_R5_DRIVE_READBACK.json")
